In [1]:
%%bash
# Download the Microsoft repository GPG keys
wget -q "https://packages.microsoft.com/config/ubuntu/$(lsb_release -rs)/packages-microsoft-prod.deb"

# Register the Microsoft repository
dpkg -i packages-microsoft-prod.deb
apt-get update -y

# Install PowerShell
apt-get install -y powershell

Selecting previously unselected package packages-microsoft-prod.
(Reading database ... 125128 files and directories currently installed.)
Preparing to unpack packages-microsoft-prod.deb ...
Unpacking packages-microsoft-prod (1.0-ubuntu22.04.1) ...
Setting up packages-microsoft-prod (1.0-ubuntu22.04.1) ...
Hit:1 http://archive.ubuntu.com/ubuntu jammy InRelease
Get:2 http://archive.ubuntu.com/ubuntu jammy-updates InRelease [128 kB]
Get:3 http://security.ubuntu.com/ubuntu jammy-security InRelease [129 kB]
Get:4 https://cli.github.com/packages stable InRelease [3,917 B]
Get:5 https://cloud.r-project.org/bin/linux/ubuntu jammy-cran40/ InRelease [3,632 B]
Get:6 https://packages.microsoft.com/ubuntu/22.04/prod jammy InRelease [3,632 B]
Get:7 http://archive.ubuntu.com/ubuntu jammy-backports InRelease [127 kB]
Get:8 https://developer.download.nvidia.com/compute/cuda/repos/ubuntu2204/x86_64  InRelease [1,581 B]
Get:9 https://ppa.launchpadcontent.net/deadsnakes/ppa/ubuntu jammy InRelease [18.1 kB

W: Skipping acquire of configured file 'main/source/Sources' as repository 'https://r2u.stat.illinois.edu/ubuntu jammy InRelease' does not seem to provide it (sources.list entry misspelt?)


In [2]:
!pip install torch_geometric

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 64.4/64.4 kB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 37.5 MB/s eta 0:00:00


In [3]:
import json
import subprocess
import hashlib
import pickle
from pathlib import Path
import pandas as pd
import numpy as np
import torch
import time
from torch_geometric.data import Data
from tqdm.notebook import tqdm

NODE_TYPE_VOCAB_CAP = 200
TOKEN_HASH_DIM = 32
NODE_FEAT_DIM = NODE_TYPE_VOCAB_CAP + TOKEN_HASH_DIM  # 232

def _hash_token_to_vec(token: str, dim: int = TOKEN_HASH_DIM) -> np.ndarray:
    if not token: return np.zeros(dim, dtype=np.float32)
    h = hashlib.sha256(token.encode('utf-8', errors='replace')).digest()
    needed = dim * 4
    if len(h) < needed: h = (h * (needed // len(h) + 1))[:needed]
    arr = np.frombuffer(h[:needed], dtype=np.int32).astype(np.float32)
    return np.tanh(arr / (2**24))

def create_ps_extractor():
    ps_code = """
    param([string]$ScriptPath)
    $ErrorActionPreference = 'SilentlyContinue'
    
    $code = [System.IO.File]::ReadAllText($ScriptPath)
    $ast = [System.Management.Automation.Language.Parser]::ParseInput($code, [ref]$null, [ref]$null)
    $allNodes = $ast.FindAll({$true}, $true)
    
    $nodeList = @()
    $edgeList = @()
    $nodeMap = @{}
    
    for ($i = 0; $i -lt $allNodes.Count; $i++) {
        $nodeMap[$allNodes[$i]] = $i
        $nodeList += $allNodes[$i].GetType().Name
    }
    
    for ($i = 0; $i -lt $allNodes.Count; $i++) {
        $parent = $allNodes[$i].Parent
        if ($null -ne $parent -and $nodeMap.ContainsKey($parent)) {
            $parentId = $nodeMap[$parent]
            $edgeList += ,@($parentId, $i)
        }
    }
    
    $result = @{ nodes = $nodeList; edges = $edgeList }
    $result | ConvertTo-Json -Depth 10 -Compress
    """
    with open("/kaggle/working/ast_extractor.ps1", "w", encoding="utf-8") as f:
        f.write(ps_code)

def generate_ast_graph(script_text: str, max_nodes: int = 800) -> Data:
    with open("/kaggle/working/temp_script.ps1", "w", encoding="utf-8") as f:
        f.write(script_text)
        
    # Notice we use 'pwsh' for Linux PowerShell
    cmd = ["pwsh", "-NoProfile", "-ExecutionPolicy", "Bypass", "-File", "/kaggle/working/ast_extractor.ps1", "/kaggle/working/temp_script.ps1"]
    status = "success" # <-- NEW
    try:
        result = subprocess.run(cmd, capture_output=True, text=True, timeout=5)
        ast_data = json.loads(result.stdout)
        node_types = ast_data.get('nodes', [])
        edges = ast_data.get('edges', [])
    except Exception:
        node_types = ['ErrorAst']
        edges = []
        status = "error" # <-- NEW

    if not node_types: 
        node_types = ['EmptyAst']
        status = "empty" if status == "success" else status # <-- NEW

    node_types = node_types[:max_nodes]
    n = len(node_types)
    valid_edges = [e for e in edges if e[0] < n and e[1] < n]
    
    type_ids = np.array([int(hashlib.md5(t.lower().encode()).hexdigest(), 16) % NODE_TYPE_VOCAB_CAP 
                         for t in node_types], dtype=np.int32)
                         
    hash_cache = np.zeros((n, TOKEN_HASH_DIM), dtype=np.float16)
    for i, t in enumerate(node_types): 
        hash_cache[i] = _hash_token_to_vec(t).astype(np.float16)
        
    x = np.zeros((n, NODE_FEAT_DIM), dtype=np.float16)
    x[np.arange(n), type_ids] = 1.0 
    x[:, NODE_TYPE_VOCAB_CAP:] = hash_cache
    
    if valid_edges:
        src = [e[0] for e in valid_edges]
        dst = [e[1] for e in valid_edges]
        edge_array = np.stack([np.concatenate([src, dst]), np.concatenate([dst, src])], axis=0)
    else:
        edge_array = np.array([[0],[0]], dtype=np.int64)
        
    data_obj = Data(x=torch.from_numpy(x.astype(np.float32)), 
                    edge_index=torch.from_numpy(edge_array.astype(np.int64)))
    return data_obj, status

# Find the splits directory in Kaggle inputs
# NEW: Setup Output Directory
OUT_DIR = Path('/kaggle/working/ast_outputs')
OUT_DIR.mkdir(parents=True, exist_ok=True)

# Find the splits directory in Kaggle inputs
splits_dir = None
for p in Path('/kaggle/input').rglob('train_seed42.parquet'):
    splits_dir = p.parent
    break

if not splits_dir:
    raise FileNotFoundError("Could not find the splits dataset. Please attach it to this notebook.")

create_ps_extractor()
graph_dict = {}
all_scripts = {}

parquet_files = list(splits_dir.glob("*.parquet"))
print(f"Found {len(parquet_files)} parquet files. Extracting unique scripts...")

for pf in parquet_files:
    df = pd.read_parquet(pf)
    for _, row in df.iterrows():
        sha256 = row['sha256']
        if sha256 not in all_scripts:
            all_scripts[sha256] = row['script_text']

print(f"Total unique scripts to parse: {len(all_scripts)}")

# NEW: Initialize stats tracker
stats = {'total': len(all_scripts), 'success': 0, 'error': 0, 'empty': 0}
t0 = time.time()

for sha256, text in tqdm(all_scripts.items(), desc="Parsing ASTs"):
    g, status = generate_ast_graph(str(text)) # <-- NEW: Unpack tuple
    graph_dict[sha256] = g
    stats[status] += 1 # <-- NEW: Track status
    
stats['elapsed_minutes'] = round((time.time() - t0) / 60, 2)

# NEW: Save to the new OUT_DIR
output_pkl = OUT_DIR / "ps_ast_graphs.pkl"
with open(output_pkl, "wb") as f:
    pickle.dump(graph_dict, f)

# NEW: Save the JSON summary
summary_json = OUT_DIR / "extraction_summary.json"
with open(summary_json, "w") as f:
    json.dump(stats, f, indent=2)
    
print(f"\nSaved AST graphs to {output_pkl}!")
print(f"Saved summary to {summary_json}!")

Found 14 parquet files. Extracting unique scripts...
Total unique scripts to parse: 17802


Parsing ASTs:   0%|          | 0/17802 [00:00<?, ?it/s]


Saved AST graphs to /kaggle/working/ast_outputs/ps_ast_graphs.pkl!
Saved summary to /kaggle/working/ast_outputs/extraction_summary.json!


In [4]:
print("\n=== AST EXTRACTION SANITY CHECKS ===")
import os
from pathlib import Path
import json

ck_ok = 0
ck_fail = 0

def chk(c, n, d=''):
    global ck_ok, ck_fail
    if c:
        ck_ok += 1
        print(f"[OK]   {n}")
    else:
        ck_fail += 1
        print(f"[FAIL] {n}  {d}")

out_dir = Path('/kaggle/working/ast_outputs')
pkl_path = out_dir / 'ps_ast_graphs.pkl'
json_path = out_dir / 'extraction_summary.json'

chk(pkl_path.exists(), "ps_ast_graphs.pkl exists")
if pkl_path.exists():
    size_mb = os.path.getsize(pkl_path) / (1024 * 1024)
    chk(size_mb > 1.0, f"PKL file size is reasonable ({size_mb:.2f} MB)")

chk(json_path.exists(), "extraction_summary.json exists")

if json_path.exists():
    with open(json_path, 'r') as f:
        stats = json.load(f)
    print("\nExtraction Summary:")
    for k, v in stats.items():
        print(f"  {k}: {v}")

print(f"\n{ck_ok} passed, {ck_fail} failed")
if ck_fail == 0:
    print("\nALL CHECKS PASSED. Ready to upload /kaggle/working/ast_outputs/ as a Dataset.")
else:
    print("\nWARNING: Some checks failed. Review the output above.")


=== AST EXTRACTION SANITY CHECKS ===
[OK]   ps_ast_graphs.pkl exists
[OK]   PKL file size is reasonable (5824.00 MB)
[OK]   extraction_summary.json exists

Extraction Summary:
  total: 17802
  success: 17596
  error: 206
  empty: 0
  elapsed_minutes: 200.45

3 passed, 0 failed

ALL CHECKS PASSED. Ready to upload /kaggle/working/ast_outputs/ as a Dataset.
